# xlsx_to_data_jsonl.ipynb

Convert a manually-labelled Excel sheet into a **DATA JSONL** file usable for:
- accuracy testing
- later generation of OpenAI fine-tuning JSONL

**Output schema per line (one example):**
```json
{
  "VIDEOTITEL": "...",
  "VIDEOCONTEXT": "...",
  "TARGETCOMMENT": "...",
  "labels": {
    "C1": 0,
    "C2": 0,
    "C3": 0,
    "C4": 0,
    "C6": 0
  },
  "metadata": {
    "link": "...",
    "commenter_account": "..."
  }
}
```

Adjust column names in the `COLUMN_MAP` cell if your XLSX differs.


In [7]:
# --- 0) Imports ---
import pandas as pd
import json
from pathlib import Path


## 1) Set paths

In [8]:
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "README.md").exists():
            return p
    raise RuntimeError("Repo root not found from start=%s" % start)

repo_root = find_repo_root()
print("Repo root:", repo_root)
# Example read


# Path to your XLSX.
# Put the file in the same folder as this notebook OR update this path.
XLSX_PATH = repo_root / "datasets/yt_factlink/00_base_data/manual_labels_386_v2.xlsx"

# Where to write JSONL
OUT_PATH = repo_root / "datasets/yt_factlink/01_conversion/02_outputs/manual_labels_386_v2.data.jsonl"

Repo root: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer


## 2) Load XLSX

In [9]:
# install missing engine for pandas.read_excel
%pip install openpyxl

df = pd.read_excel(XLSX_PATH, skiprows=1)
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Rows: 386
Columns: ['video_title', 'link', 'context', 'commenter_account', 'Comments', 'c1_manlab', 'c2_manlab', 'c3_manlab', 'c4_manlab', 'c6_manlab']


,video_title,link,context,commenter_account,Comments,c1_manlab,c2_manlab,c3_manlab,c4_manlab,c6_manlab
0,"The Chinese representative angrily said ""Taiwa...",https://www.youtube.com/watch?v=hSXQg2IQ3QM,At the 78th Special Session of the UN General ...,@小茹陳-q8y,和平不應建立在威脅之上，謝謝小野之問，說出台灣人心聲❤,0,0,0,0,0
1,"The Chinese representative angrily said ""Taiwa...",NaN,At the 78th Special Session of the UN General ...,@lesimkien724,(hand-purple-blue-peace),0,0,0,0,0
2,"The Chinese representative angrily said ""Taiwa...",NaN,At the 78th Special Session of the UN General ...,@maybo687\n,日本的司馬昭之心、人盡皆知、只有台灣的35歲以下的年輕人不知、,0,0,0,0,0
3,"The Chinese representative angrily said ""Taiwa...",NaN,At the 78th Special Session of the UN General ...,\n@taiwaingo6218,仇視台灣的不是日本國和日本人！你明白嗎？一個國家仇恨自己的國人！你相信這國家是真心愛國人嗎？,0,0,1,0,0
4,"The Chinese representative angrily said ""Taiwa...",NaN,At the 78th Special Session of the UN General ...,@shihongliao,因为台湾现管理者一直鼓动利用增强各种武力阻止和平统一，更有外部国家在台湾武力的支撑，,0,0,1,0,0


## 3) Column mapping

Columns:
- `video_title`
- `link`
- `context`
- `commenter_account`
- `Comments`
- `c1_manlab`, `c2_manlab`, `c3_manlab`, `c4_manlab`, `c6_manlab`

We map them into the canonical keys:
- `VIDEOTITEL`
- `VIDEOCONTEXT`
- `TARGETCOMMENT`

If your file uses slightly different names, edit `COLUMN_MAP` below.


In [10]:
COLUMN_MAP = {
    # canonical_key : xlsx_column_name
    "VIDEOTITEL": "video_title",
    "VIDEOCONTEXT": "context",
    "TARGETCOMMENT": "Comments",

    # optional metadata
    "link": "link",
    "commenter_account": "commenter_account",

    # labels (manual)
    "C1": "c1_manlab",
    "C2": "c2_manlab",
    "C3": "c3_manlab",
    "C4": "c4_manlab",
    "C6": "c6_manlab",
}

# quick check for missing columns
missing = [v for v in COLUMN_MAP.values() if v not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns in XLSX: {missing}")
print("Column map OK.")


Column map OK.


## 4) Clean rows + build JSONL records

In [11]:
def clean_text(x):
    if pd.isna(x):
        return ""
    # force string, strip whitespace
    return str(x).strip()

def clean_label(x):
    if pd.isna(x):
        return 0
    # allow "1/0", 1/0, True/False, etc.
    s = str(x).strip()
    if s in ("1", "1.0", "True", "true", "yes", "YES"):
        return 1
    return 0

records = []
for i, (_, row) in enumerate(df.iterrows()):
    rec = {
        "id": f"syn_{i:04d}",       # stable synthetic ID
        "VIDEOTITEL": clean_text(row[COLUMN_MAP["VIDEOTITEL"]]),
        "VIDEOCONTEXT": clean_text(row[COLUMN_MAP["VIDEOCONTEXT"]]),
        "TARGETCOMMENT": clean_text(row[COLUMN_MAP["TARGETCOMMENT"]]),
        "labels": {
            "C1": clean_label(row[COLUMN_MAP["C1"]]),
            "C2": clean_label(row[COLUMN_MAP["C2"]]),
            "C3": clean_label(row[COLUMN_MAP["C3"]]),
            "C4": clean_label(row[COLUMN_MAP["C4"]]),
            "C6": clean_label(row[COLUMN_MAP["C6"]]),
        },
        "metadata": {
            "link": clean_text(row[COLUMN_MAP["link"]]) if COLUMN_MAP.get("link") else "",
            "commenter_account": clean_text(row[COLUMN_MAP["commenter_account"]]) if COLUMN_MAP.get("commenter_account") else "",
        }
    }

    # drop rows with empty comment (usually useless)
    if rec["TARGETCOMMENT"]:
        records.append(rec)

print("Usable records:", len(records))
records[0]


Usable records: 386


{'id': 'syn_0000',
 'VIDEOTITEL': 'The Chinese representative angrily said "Taiwan belongs to China"!',
 'VIDEOCONTEXT': "At the 78th Special Session of the UN General Assembly, China's representative forcefully asserted the one-China principle, claiming Taiwan as inalienable sovereign territory and warning against independence. Japan's representative, Ono Kosei, countered calmly by asking a profound question: if Taiwan truly belongs to China, why is military force necessary for unification? He urged the assembly to respect the principle of self-determination and the wishes of the 23 million Taiwanese people. This thoughtful challenge disrupted the aggressive momentum, sparking a moment of silence and later earning support from U.S. and European delegates. Following the public impasse, the Chinese and Japanese diplomats shared a rare private moment, candidly discussing their difficult positions and expressing a mutual hope for future peace. The entire incident rekindled global debate o

## 5) Write DATA JSONL

In [12]:
with OUT_PATH.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("Wrote:", OUT_PATH.resolve())


Wrote: /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/ici01_PublicOpinionAnalyzer/datasets/yt_factlink/01_conversion/02_outputs/manual_labels_386_v2.data.jsonl


## 6) Sanity check

Read the first few lines back to confirm format.


In [13]:
def read_jsonl(path, n=3):
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            out.append(json.loads(line))
            if i+1 >= n:
                break
    return out

read_jsonl(OUT_PATH, 3)


[{'id': 'syn_0000',
  'VIDEOTITEL': 'The Chinese representative angrily said "Taiwan belongs to China"!',
  'VIDEOCONTEXT': "At the 78th Special Session of the UN General Assembly, China's representative forcefully asserted the one-China principle, claiming Taiwan as inalienable sovereign territory and warning against independence. Japan's representative, Ono Kosei, countered calmly by asking a profound question: if Taiwan truly belongs to China, why is military force necessary for unification? He urged the assembly to respect the principle of self-determination and the wishes of the 23 million Taiwanese people. This thoughtful challenge disrupted the aggressive momentum, sparking a moment of silence and later earning support from U.S. and European delegates. Following the public impasse, the Chinese and Japanese diplomats shared a rare private moment, candidly discussing their difficult positions and expressing a mutual hope for future peace. The entire incident rekindled global debat